In [0]:
from pyspark.sql.functions import current_timestamp, col
from datetime import datetime

date = datetime.now().strftime("%Y%m%d")

CATALOG_NAME = "isp"
SCHEMA_NAME = "bronze"
VOLUME_NAME = "isp_volumes"
FOLDER_NAME = "raw"
FILE_NAME = f"timeseries_monthly_since_2003_{date}.csv"
TABLE_NAME = f"timeseries_monthly_since_2003"

source_volume = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}/{FOLDER_NAME}"
target_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"
checkpoint_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}/_checkpoints/{TABLE_NAME}"

In [0]:
df_stream = (
    spark.readStream
    .format("cloudfiles")
    .option("cloudfiles.format", "csv")
    .option("cloudfiles.schemaLocation", f"{checkpoint_path}/schema")
    .option("header", "true")
    .option("delimiter", ";")
    .option("cloudfiles.inferColumnTypes", "true")
    .option("pathGlobFilter", FILE_NAME)
    .load(source_volume)
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

In [0]:
query = (
    df_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{checkpoint_path}/data")
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
%sql

SELECT * FROM isp.bronze.timeseries_monthly_since_2003